In [ ]:
import pandas as pd
import os

#1. Defina o caminho exato da rede onde está o arquivo bruto da empresa
caminho_rede = "dados"
nome_arquivo = "dados_importacao_simulados.xlsx"

caminho_completo = r"C:\Users\gustavo.lourenco\Downloads\dados_importacao_simulados.xlsx"

# 2. Tente ler o arquivo
print("Tentando conectar e ler o arquivo da rede...")
df = pd.read_excel(caminho_completo, sheet_name="BASE")

# 3. Exiba as primeiras 5 linhas para garantir que funcionou
df.head()

Tentando conectar e ler o arquivo da rede...


,PAÍS,Data,U$/KG,Peso líquido,NCM,Notificado_Final,ID_ITEM,Descrição produto,MODAL,PROVÁVEL IMPORTADOR,PROVÁVEL EXPORTADOR,VALOR FOB ESTIMADO TOTAL
0,WAKANDA,2026-01-01,13.782383,3860,3920,HIGHTECHCORP,ALVEOLAR,"""info"",""vidro"",""filme""",MARITIMA,HIGHTECHCORP,HOLANDA BAIXO,53200
1,WAKANDA,2026-01-01,14.228916,4150,3920,POLI-INDUSTRIA-INDUSTRIAL,COMPACTA,"""info"",""operador"",""vidro""",MARITIMA,HIGHTECHCORP,STATES INVESTIMENTOS,59050
2,MORIOH,2026-01-01,1.181818,13200,3920,ROBLOX,ALVEOLAR,"""info"",""operador"",""vidro""",MARITIMA,ROBLOX,ORIGINAL CHINA,15600
3,WAKANDA,2026-01-01,11.586072,16399,3920,HIGHTECHCORP,ALVEOLAR,"""info"",""peça"",""computador""",MARITIMA,HIGHTECHCORP,STATES INVESTIMENTOS,190000
4,MORIOH,2026-01-01,9.547905,21000,3920,DAY JAPÃO,COMPACTA,"""info"",""chapa"",""comp""",RODOVIARIA,HIGHTECHCORP,ORIGINAL CHINA,200506


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   PAÍS                      20 non-null     object        
 1   Data                      20 non-null     datetime64[ns]
 2   U$/KG                     20 non-null     float64       
 3   Peso líquido              20 non-null     int64         
 4   NCM                       20 non-null     int64         
 5   Notificado_Final          20 non-null     object        
 6   ID_ITEM                   20 non-null     object        
 7   Descrição produto         20 non-null     object        
 8   MODAL                     20 non-null     object        
 9   PROVÁVEL IMPORTADOR       20 non-null     object        
 10  PROVÁVEL EXPORTADOR       20 non-null     object        
 11  VALOR FOB ESTIMADO TOTAL  20 non-null     int64         
dtypes: datetime64[ns](1), fl

In [26]:
# 1. Converter a coluna 'Data' para o formato de data real do Python
# (Substitua 'Data' pelo nome exato com maiúscula/minúscula da sua coluna)
df['Data'] = pd.to_datetime(df['Data'], dayfirst=True, errors='coerce')

# 2. Criar colunas auxiliares de ANO e MÊS automaticamente
df['ANO'] = df['Data'].dt.year
df['MÊS'] = df['Data'].dt.month

# 2. Filtrar a base de 2026 apenas pelos produtos que importam
produtos_alvo = ["ALVEOLAR", "COMPACTA"]
df_2026_filtrado = df[(df['ANO'] == 2026) & (df['ID_ITEM'].isin(produtos_alvo))]

# 3. Criar um DICIONÁRIO para separar e ordenar o Top 5 de cada um
top_5_por_produto = {}
todas_empresas_top = []

for prod in produtos_alvo:
    # Filtra as linhas do produto atual
    df_prod = df_2026_filtrado[df_2026_filtrado['ID_ITEM'] == prod]
    
    # Agrupa, soma e pega as 5 maiores (já vem em ordem decrescente do maior para o menor)
    top_5_ordenado = df_prod.groupby('Notificado_Final')['VALOR FOB ESTIMADO TOTAL'].sum().nlargest(5).index.tolist()
    
    # Salva no dicionário para você consultar depois quem é quem
    top_5_por_produto[prod] = top_5_ordenado
    
    # Alimenta a lista geral que usaremos para filtrar a base histórica
    todas_empresas_top.extend(top_5_ordenado)

# Remove duplicadas caso uma empresa esteja no ranking dos dois produtos
empresas_finais_unicas = list(set(todas_empresas_top))

# --- VISUALIZAÇÃO DOS RESULTADOS NO NOTEBOOK ---
print("--- RANKING DE ALVEOLAR (Ordem Decrescente): ---")
print(top_5_por_produto["ALVEOLAR"])

print("\n--- RANKING DE COMPACTA (Ordem Decrescente): ---")
print(top_5_por_produto["COMPACTA"])

--- RANKING DE ALVEOLAR (Ordem Decrescente): ---
['HIGHTECHCORP', 'ROBLOX']

--- RANKING DE COMPACTA (Ordem Decrescente): ---
['DAY JAPÃO', 'POLI-INDUSTRIA-INDUSTRIAL']


In [27]:
import plotly.express as px

# 1. Filtrar a base mãe trazendo dados históricos (2023 - 2026) APENAS das empresas do Top 5
# Corrigido o filtro de anos para [2023 - 2026]
df_historico = df[
    (df['Notificado_Final'].isin(empresas_finais_unicas)) & 
    (df['ANO'].isin([2023, 2024, 2025, 2026])) &
    (df['ID_ITEM'].isin(["ALVEOLAR", "COMPACTA"]))
]

# 2. Agrupar os dados por Empresa, Ano e Mês para estruturar o gráfico
df_grafico = (
    df_historico.groupby(['ID_ITEM', 'Notificado_Final', 'ANO', 'MÊS'])['VALOR FOB ESTIMADO TOTAL']
    .sum()
    .reset_index()
)

# Garantir que o Ano seja tratado como texto para o gráfico criar barras separadas (lado a lado)
df_grafico['ANO'] = df_grafico['ANO'].astype(str)

# 3. Criar o Gráfico Interativo com o Plotly
fig = px.bar(
    df_grafico, 
    x="MÊS", 
    y="VALOR FOB ESTIMADO TOTAL", 
    color="ANO",                 # Cria barras lado a lado para 2023 até 2026
    barmode="group",             # Força as barras a ficarem juntas/emparelhadas por mês
    facet_col="Notificado_Final",# Cria um minigráfico separado para cada empresa
    facet_row="ID_ITEM",         # Separa Alveolar (linha de cima) de Compacta (linha de baixo)
    title="Atuação dos Concorrentes Top 5 (2023 - 2026) ao Longo dos Anos",
    labels={"VALOR FOB ESTIMADO TOTAL": "Valor FOB (USD)", "MÊS": "Mês"}
)

# Ajustes estéticos para legibilidade
fig.update_xaxes(dtick=1) # Garante que mostre os meses de 1 a 12 certinho
fig.update_layout(height=700, width=1200)

# Exibir o gráfico na tela do Notebook
fig.show()

In [28]:
import plotly.express as px

paleta_cores_empresas = ["#1F4E78", "#D9531E", "#2CA02C", "#9467BD", "#8C564B"]

fig = px.bar(
    df_grafico, 
    x="MÊS", 
    y="VALOR FOB ESTIMADO TOTAL", 
    color="Notificado_Final",     
    pattern_shape="ANO",          
    barmode="group",              
    facet_row="ID_ITEM",          
    title="Análise Comparativa de Concorrentes (2023 - 2026) por Ano",
    labels={
        "VALOR FOB ESTIMADO TOTAL": "Valor FOB (USD)", 
        "MÊS": "Mês", 
        "Notificado_Final": "Empresa",
        "ANO": "Ano"
    },
    color_discrete_sequence=paleta_cores_empresas
)

# --- A MÁGICA PARA LIMPAR A LEGENDA REPETIDA ---
nomes_visualizados = set()
fig.for_each_trace(lambda trace: (
    # Pega apenas o nome da empresa antes da vírgula
    trace.update(name=trace.name.split(',')[0], legendgroup=trace.name.split(',')[0]),
    # Se a empresa já apareceu, esconde a repetição do ano seguinte na legenda
    trace.update(showlegend=False) if trace.name.split(',')[0] in nomes_visualizados else nomes_visualizados.add(trace.name.split(',')[0])
))

# Limpar os títulos das linhas ("ID_ITEM=")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=12)))

# Forçar o Eixo X a exibir APENAS os meses que possuem dados na tabela (evita o buraco vazio)
meses_com_dados = sorted(df_grafico['MÊS'].unique())
fig.update_xaxes(
    tickmode="array",
    tickvals=meses_com_dados,
    ticktext=[str(m) for m in meses_com_dados],
    tickangle=0,
    tickfont=dict(size=11)
)

fig.update_yaxes(tickprefix="$", tickformat=",", showgrid=True, gridcolor="#E5E5E5")

fig.update_layout(
    height=650,          
    width=1200,          
    margin=dict(t=80, b=50, l=80, r=40),
    title_font=dict(size=18, family="Arial"),
    plot_bgcolor="white"
)

fig.show()